# HW03-ICA :: Part B (Transfer Learning + Fine-Tuning with ResNet50)

COSC 6373

Adam Nelson-Archer, 2140122

## Goal

Use **transfer learning** with a pre-trained **ResNet50 (ImageNet weights)** to classify **horses vs. camels**.

Required steps for Part B:
1. Load the pre-trained base model and weights.
2. Train the model using weight fine-tuning.
3. Evaluate performance and report the confusion matrix.
4. State which layers were kept/replaced, and report the best architecture you found (based on your run).

## Prerequisites / Dataset layout

This notebook expects the dataset in:

- `HW3/dataset/train/<class_name>/*.jpg`
- `HW3/dataset/test/<class_name>/*.jpg`

Where `<class_name>` are the two folders for the classes (e.g., `horses` and `camels`, or similar).

If you used Part A already, keep the same directory structure.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Reproducibility (best-effort)
SEED = 6373
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("CWD:", os.getcwd())

In [ ]:
# Paths (relative to this notebook: HW3/Part_B)
NOTEBOOK_DIR = Path.cwd()
DATASET_DIR = (NOTEBOOK_DIR / ".." / "dataset").resolve()
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# ResNet50 default input size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("DATASET_DIR =", DATASET_DIR)
print("TRAIN_DIR   =", TRAIN_DIR)
print("TEST_DIR    =", TEST_DIR)
print("Train exists:", TRAIN_DIR.exists())
print("Test exists:", TEST_DIR.exists())

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        "Dataset folders not found. Expected:\n"
        f"- {TRAIN_DIR}\n"
        f"- {TEST_DIR}\n\n"
        "Create these folders and place class subfolders inside them, e.g.:\n"
        "- HW3/dataset/train/horses\n"
        "- HW3/dataset/train/camels\n"
        "- HW3/dataset/test/horses\n"
        "- HW3/dataset/test/camels\n"
    )

In [ ]:
# Load datasets
# We'll create a validation split from TRAIN_DIR for model selection.
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="binary",
    shuffle=False,  # important for confusion matrix alignment
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Class names (alphabetical):", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Visualize a few training images
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"label={int(labels[i].numpy()[0])}")
        plt.axis("off")
plt.tight_layout()